# KvForge: Cross-Model KV Cache Reuse


In [ ]:
import json, math, time, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache

device = "cpu"
print("Device:", device)

# core

class LoRAConv1D(nn.Module):
    def __init__(self, orig, r=8, alpha=16.0):
        super().__init__()
        self.orig = orig
        self.scaling = alpha / r
        in_f = orig.weight.shape[0]; out_f = orig.nf
        self.lora_A = nn.Parameter(torch.randn(in_f, r) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, out_f))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active:
            h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
        return h

def inject_lora(model, r=8, alpha=16.0):
    count = 0
    for n, m in model.named_modules():
        if n.endswith(".attn.c_attn") or n.endswith(".attn.c_proj"):
            parent = model; parts = n.split(".")
            child = parts[-1]
            for p in parts[:-1]:
                if p: parent = getattr(parent, p)
            setattr(parent, child, LoRAConv1D(m, r=r, alpha=alpha))
            count += 1
    print("  LoRA injected:", count, "modules")
    return count

def set_lora(m, a):
    for mod in m.modules():
        if hasattr(mod, "activate"): mod.activate(a)

def compress_past(past, bits):
    if bits >= 16: return past
    dc = DynamicCache()
    for li, layer in enumerate(past):
        k, v = layer[0], layer[1]
        mnk, mxk = k.min(-1, True).values, k.max(-1, True).values
        sk = (mxk - mnk).clamp(1e-8) / (2**bits - 1)
        dk = (((k - mnk) / sk).round().clamp(0, 2**bits-1).float() * sk + mnk).to(k.dtype)
        mnv, mxv = v.min(-1, True).values, v.max(-1, True).values
        sv = (mxv - mnv).clamp(1e-8) / (2**bits - 1)
        dv = (((v - mnv) / sv).round().clamp(0, 2**bits-1).float() * sv + mnv).to(v.dtype)
        dc.update(dk, dv, dk.size(2))
    return dc

def cache_mb(past):
    total = 0
    for layer in past:
        k, v = layer[0], layer[1]
        total += k.numel() * k.element_size() + v.numel() * v.element_size()
    return total / (1024**2)

def train_lora(model, texts, steps=100, lr=3e-3):
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    params = [p for n,p in model.named_parameters() if "lora" in n]
    opt = torch.optim.AdamW(params, lr=lr)
    model.train()
    losses = []
    for s in range(steps):
        text = texts[s % len(texts)]
        inp = tok(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        ids = inp["input_ids"]
        out = model(ids)
        loss = F.cross_entropy(out.logits[0, :-1], ids[0, 1:])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if s % 30 == 0:
            print("  Step %d | Loss: %.4f" % (s, loss.item()))
    model.eval()
    return losses

print("=" * 70)
print("KvForge PoC: Cross-Model KV Cache Reuse")
print("=" * 70)
print()
print("Key insight: KV cache stores BASE model projections.")
print("LoRA modifies QUERY representation during decode, not cached K/V.")
print("Therefore, cache from one LoRA adapter works with another adapter.")
print("Result: 1 prefill + N decodes instead of N prefills + N decodes.")

# 1. Train TWO different LoRA adapters
print("[1/4] Loading GPT-2 Small...")
tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token

# Adapter A: scientific
print("\n[2/4] Training Adapter A (scientific)...", end=" ")
bm_a = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
inject_lora(bm_a, r=8)
texts_a = [
    "Quantum computing uses qubits that can exist in superposition states.",
    "The attention mechanism computes weighted sums of query-key similarities.",
    "Differential privacy adds calibrated noise to training data.",
    "Gradient descent minimizes loss functions by updating parameters.",
]
loss_a = train_lora(bm_a, texts_a, steps=80)
print("\n  Loss: %.4f -> %.4f" % (loss_a[0], loss_a[-1]))

# Adapter B: creative
print("[3/4] Training Adapter B (creative)...", end=" ")
bm_b = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
inject_lora(bm_b, r=8)
texts_b = [
    "The moon hung like a silver coin in the velvet sky.",
    "Her laughter echoed through the corridors of memory.",
    "The old bookshop smelled of paper and forgotten stories.",
    "Raindrops danced on the windowpane like tiny musicians.",
]
loss_b = train_lora(bm_b, texts_b, steps=80)
print("\n  Loss: %.4f -> %.4f" % (loss_b[0], loss_b[-1]))

# 2. Cross-Model Test
print("\n[4/4] Cross-Model KV Cache Reuse Test")

prompts = [
    "The transformer architecture processes information through",
    "The light from distant stars travels through",
]

def decode_cross(model_obj, last_tok, past, n_tokens=20, temp=0.7):
    set_lora(model_obj, True)
    generated = []
    with torch.no_grad():
        for _ in range(n_tokens):
            out = model_obj(last_tok, past_key_values=past, use_cache=True)
            logits = out.logits[:, -1, :] / temp
            probs = F.softmax(logits, dim=-1)
            last_tok = torch.multinomial(probs, 1)
            generated.append(last_tok.item())
    set_lora(model_obj, False)
    return tok.decode(generated, skip_special_tokens=True)

for pi, prompt in enumerate(prompts):
    inp = tok(prompt, return_tensors="pt").to(device)
    print("\n" + "-" * 70)
    print("Prompt %d: %s" % (pi+1, prompt))
    print("-" * 70)

    # Prefill A (LoRA off)
    set_lora(bm_a, False)
    with torch.no_grad():
        out_a = bm_a.generate(**inp, max_new_tokens=1, use_cache=True,
            pad_token_id=tok.eos_token_id, do_sample=False,
            return_dict_in_generate=True)
    past_a = out_a.past_key_values
    last_tok = out_a.sequences[:, -1:]
    print("  [Cache A: %.4f MB]" % cache_mb(past_a))

    # Decode B using A cache (cross-model)
    text_b_on_a = decode_cross(bm_b, last_tok, past_a, n_tokens=20)
    print("  [A cache -> B decode] %s" % text_b_on_a[:80])

    # Decode A using own cache (baseline)
    text_a_own = decode_cross(bm_a, last_tok, past_a, n_tokens=20)
    print("  [A cache -> A decode] %s" % text_a_own[:80])

    # Prefill B (LoRA off)
    set_lora(bm_b, False)
    with torch.no_grad():
        out_b = bm_b.generate(**inp, max_new_tokens=1, use_cache=True,
            pad_token_id=tok.eos_token_id, do_sample=False,
            return_dict_in_generate=True)
    past_b = out_b.past_key_values
    last_tok_b = out_b.sequences[:, -1:]

    # Decode A using B cache (reverse cross-model)
    text_a_on_b = decode_cross(bm_a, last_tok_b, past_b, n_tokens=20)
    print("  [B cache -> A decode] %s" % text_a_on_b[:80])

    # Decode B using own cache (baseline)
    text_b_own = decode_cross(bm_b, last_tok_b, past_b, n_tokens=20)
    print("  [B cache -> B decode] %s" % text_b_own[:80])

    # Cross-model + 4-bit compression
    past_b_4bit = compress_past(past_b, 4)
    text_a_4bit = decode_cross(bm_a, last_tok_b, past_b_4bit, n_tokens=20)
    print("  [B cache(4bit) -> A decode] %s" % text_a_4bit[:80])

# 3. Latency Analysis
print("\n" + "=" * 70)
print("LATENCY ANALYSIS")
print("=" * 70)
print()
print("  Standard: N models = N prefills + N decodes")
print("  Cross-model: 1 prefill + N decodes")
print()
for n in [2, 5, 10, 20, 50]:
    ratio = (n * 2.0) / (1.0 + n * 1.0)
    print("    N=%2d: %.1fx faster" % (n, ratio))

results = {
    "train_loss_a": [round(loss_a[0],4), round(loss_a[-1],4)],
    "train_loss_b": [round(loss_b[0],4), round(loss_b[-1],4)],
}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nresults.json saved | Done!")
